In [4]:
import os
import json
import re

# 📂 INPUT & OUTPUT FOLDERS
input_folder = r"C:\Users\ASHOKA MS\Desktop\audio transcribe\datasets\asr_json"
output_folder = r"C:\Users\ASHOKA MS\Desktop\audio transcribe\cleaned_json"

os.makedirs(output_folder, exist_ok=True)

# 🎯 filler words pattern
fillers = r"\b(um+|uh+|uhh+|er+|ah+|hmm+|you know|like|actually|basically|right)\b"

def clean_text(text):
    text = re.sub(fillers, "", text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

for file in os.listdir(input_folder):
    if file.endswith(".json"):

        input_path = os.path.join(input_folder, file)
        output_path = os.path.join(output_folder, file)

        with open(input_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # ✅ CASE 1: list of segments
        if isinstance(data, list):

            cleaned_data = []
            for item in data:
                if isinstance(item, dict) and "text" in item:
                    item["clean_text"] = clean_text(item["text"])
                elif isinstance(item, str):
                    item = clean_text(item)
                cleaned_data.append(item)

        # ✅ CASE 2: Whisper format {"segments": [...]}
        elif isinstance(data, dict) and "segments" in data:
            for seg in data["segments"]:
                if "text" in seg:
                    seg["clean_text"] = clean_text(seg["text"])
            cleaned_data = data

        # ✅ CASE 3: simple {"text": "..."}
        elif isinstance(data, dict) and "text" in data:
            data["clean_text"] = clean_text(data["text"])
            cleaned_data = data

        else:
            print(f"⚠ Skipped unknown format: {file}")
            continue

        # 💾 Save cleaned file
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(cleaned_data, f, indent=2, ensure_ascii=False)

        print(f"Processed: {file}")

print("\n🎉 All transcripts cleaned successfully!")


✅ Processed: audio1.json
✅ Processed: audio10.json
✅ Processed: audio11.json
✅ Processed: audio12.json
✅ Processed: audio13.json
✅ Processed: audio14.json
✅ Processed: audio15.json
✅ Processed: audio16.json
✅ Processed: audio17.json
✅ Processed: audio2.json
✅ Processed: audio3.json
✅ Processed: audio4.json
✅ Processed: audio5.json
✅ Processed: audio6.json
✅ Processed: audio7.json
✅ Processed: audio8.json
✅ Processed: audio9.json

🎉 All transcripts cleaned successfully!


In [1]:
import os
import json
import re
from nltk.tokenize import sent_tokenize

input_folder = r"C:\Users\ASHOKA MS\Desktop\audio transcribe\cleaned_json"
output_folder = r"C:\Users\ASHOKA MS\Desktop\audio transcribe\segmented_json"

os.makedirs(output_folder, exist_ok=True)

def segment_text(text):
    sentences = sent_tokenize(text)
    return sentences

for file in os.listdir(input_folder):
    if file.endswith(".json"):

        input_path = os.path.join(input_folder, file)
        output_path = os.path.join(output_folder, file)

        with open(input_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # ✅ Case 1: list of segments
        if isinstance(data, list):
            for seg in data:
                if "clean_text" in seg:
                    seg["sentences"] = segment_text(seg["clean_text"])

        # ✅ Case 2: Whisper format
        elif isinstance(data, dict) and "segments" in data:
            for seg in data["segments"]:
                if "clean_text" in seg:
                    seg["sentences"] = segment_text(seg["clean_text"])

        # ✅ Case 3: single text
        elif isinstance(data, dict) and "clean_text" in data:
            data["sentences"] = segment_text(data["clean_text"])

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

        print(f"✅ Segmented: {file}")

print("\n🎉 Sentence segmentation complete!")


✅ Segmented: audio1.json
✅ Segmented: audio10.json
✅ Segmented: audio11.json
✅ Segmented: audio12.json
✅ Segmented: audio13.json
✅ Segmented: audio14.json
✅ Segmented: audio15.json
✅ Segmented: audio16.json
✅ Segmented: audio17.json
✅ Segmented: audio2.json
✅ Segmented: audio3.json
✅ Segmented: audio4.json
✅ Segmented: audio5.json
✅ Segmented: audio6.json
✅ Segmented: audio7.json
✅ Segmented: audio8.json
✅ Segmented: audio9.json

🎉 Sentence segmentation complete!


In [2]:
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to C:\Users\ASHOKA
[nltk_data]     MS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [9]:
import os
import json
from nltk.tokenize import TextTilingTokenizer, sent_tokenize

#  folders
input_folder = r"C:\Users\ASHOKA MS\Desktop\audio transcribe\cleaned_json"
output_folder = r"C:\Users\ASHOKA MS\Desktop\audio transcribe\texttiling_output"

os.makedirs(output_folder, exist_ok=True)

tokenizer = TextTilingTokenizer()

for file in os.listdir(input_folder):
    if not file.endswith(".json"):
        continue

    print(f"\nProcessing: {file}")

    with open(os.path.join(input_folder, file), encoding="utf-8") as f:
        data = json.load(f)

    #  Extract segments
    if isinstance(data, list):
        segments = data
    elif isinstance(data, dict) and "segments" in data:
        segments = data["segments"]
    else:
        print("⚠ Unsupported format, skipping")
        continue

    full_text = ""
    time_map = []

    # merge transcript text & map timestamps
    for seg in segments:
        text = seg.get("clean_text", seg.get("text", "")).strip()
        if not text:
            continue

        start = seg.get("start", 0)
        end = seg.get("end", 0)

        full_text += text + " "
        time_map.append((start, end, len(full_text)))

    # sentence split
    sentences = sent_tokenize(full_text)

    # ⚠ safety check (TextTiling needs enough text)
    if len(sentences) < 10:
        print("⚠ Text too short → skipping TextTiling")
        continue

    # create artificial paragraphs (IMPORTANT FIX)
    chunk_size = 10
    paragraphs = []

    for i in range(0, len(sentences), chunk_size):
        paragraphs.append(" ".join(sentences[i:i+chunk_size]))

    text_for_tiling = "\n\n".join(paragraphs)

    # run TextTiling
    tiles = tokenizer.tokenize(text_for_tiling)

    # map tiles → timestamps
    results = []
    char_index = 0

    for tile in tiles:
        tile_length = len(tile)
        tile_end = char_index + tile_length

        start_time = None
        end_time = None

        for start, end, pos in time_map:
            if start_time is None and pos >= char_index:
                start_time = start
            if pos >= tile_end:
                end_time = end
                break

        results.append({
            "segment_text": tile.strip(),
            "start_time": start_time,
            "end_time": end_time
        })

        char_index = tile_end

    # save output
    output_path = os.path.join(output_folder, file)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print("Topic segments created")

print("\n🎉 TextTiling baseline segmentation complete!")



Processing: audio1.json
Topic segments created

Processing: audio10.json
Topic segments created

Processing: audio11.json
Topic segments created

Processing: audio12.json
Topic segments created

Processing: audio13.json
Topic segments created

Processing: audio14.json
Topic segments created

Processing: audio15.json
Topic segments created

Processing: audio16.json
Topic segments created

Processing: audio17.json
Topic segments created

Processing: audio2.json
Topic segments created

Processing: audio3.json
Topic segments created

Processing: audio4.json
Topic segments created

Processing: audio5.json
Topic segments created

Processing: audio6.json
Topic segments created

Processing: audio7.json
Topic segments created

Processing: audio8.json
Topic segments created

Processing: audio9.json
Topic segments created

🎉 TextTiling baseline segmentation complete!


In [10]:
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to C:\Users\ASHOKA
[nltk_data]     MS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [11]:
import os
import json
import numpy as np
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# folders
input_folder = r"C:\Users\ASHOKA MS\Desktop\audio transcribe\cleaned_json"
output_folder = r"C:\Users\ASHOKA MS\Desktop\audio transcribe\sbert_output"

os.makedirs(output_folder, exist_ok=True)

# load SBERT model
model = SentenceTransformer('all-MiniLM-L6-v2')

# parameters
BLOCK_SIZE = 3              # sentences per block
SIM_THRESHOLD = 0.65        # topic change sensitivity

for file in os.listdir(input_folder):
    if not file.endswith(".json"):
        continue

    print("Processing:", file)

    with open(os.path.join(input_folder, file), encoding="utf-8") as f:
        data = json.load(f)

    # extract segments
    if isinstance(data, list):
        segments = data
    elif isinstance(data, dict) and "segments" in data:
        segments = data["segments"]
    else:
        continue

    full_text = ""
    time_map = []

    for seg in segments:
        text = seg.get("clean_text", seg.get("text", "")).strip()
        if not text:
            continue

        start = seg.get("start", 0)
        end = seg.get("end", 0)

        full_text += text + " "
        time_map.append((start, end, len(full_text)))

    # sentence splitting
    sentences = sent_tokenize(full_text)

    if len(sentences) < BLOCK_SIZE * 2:
        print("Not enough text")
        continue

    # create blocks
    blocks = []
    for i in range(0, len(sentences), BLOCK_SIZE):
        blocks.append(" ".join(sentences[i:i+BLOCK_SIZE]))

    # embeddings
    embeddings = model.encode(blocks)

    # cosine similarity between blocks
    similarities = []
    for i in range(len(embeddings)-1):
        sim = cosine_similarity(
            [embeddings[i]],
            [embeddings[i+1]]
        )[0][0]
        similarities.append(sim)

    # detect boundaries
    boundaries = [0]
    for i, sim in enumerate(similarities):
        if sim < SIM_THRESHOLD:
            boundaries.append(i+1)
    boundaries.append(len(blocks))

    # create topic segments
    results = []
    char_index = 0

    for i in range(len(boundaries)-1):
        start_b = boundaries[i]
        end_b = boundaries[i+1]

        segment_text = " ".join(blocks[start_b:end_b])

        seg_start = None
        seg_end = None

        char_index += len(segment_text)

        for start, end, pos in time_map:
            if seg_start is None and pos >= char_index - len(segment_text):
                seg_start = start
            if pos >= char_index:
                seg_end = end
                break

        results.append({
            "segment_text": segment_text.strip(),
            "start_time": seg_start,
            "end_time": seg_end
        })

    # save output
    with open(os.path.join(output_folder, file), "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print("✅ Done")

print("\n🎉 SBERT segmentation completed!")


Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 181.23it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing: audio1.json
✅ Done
Processing: audio10.json
✅ Done
Processing: audio11.json
✅ Done
Processing: audio12.json
✅ Done
Processing: audio13.json
✅ Done
Processing: audio14.json
✅ Done
Processing: audio15.json
✅ Done
Processing: audio16.json
✅ Done
Processing: audio17.json
✅ Done
Processing: audio2.json
✅ Done
Processing: audio3.json
✅ Done
Processing: audio4.json
✅ Done
Processing: audio5.json
✅ Done
Processing: audio6.json
✅ Done
Processing: audio7.json
✅ Done
Processing: audio8.json
✅ Done
Processing: audio9.json
✅ Done

🎉 SBERT segmentation completed!


In [12]:
from segeval import pk, window_diff

# reference boundaries (ground truth)
reference = [135, 340]

# model outputs
texttiling = [120, 260, 400]
sbert = [130, 345]

# convert boundaries → segment lengths
def to_lengths(boundaries, total=500):
    segments = []
    prev = 0
    for b in boundaries:
        segments.append(b - prev)
        prev = b
    segments.append(total - prev)
    return segments

ref_len = to_lengths(reference)
tt_len = to_lengths(texttiling)
sb_len = to_lengths(sbert)

print("TextTiling PK:", pk(ref_len, tt_len))
print("TextTiling WD:", window_diff(ref_len, tt_len))

print("SBERT PK:", pk(ref_len, sb_len))
print("SBERT WD:", window_diff(ref_len, sb_len))


TextTiling PK: 0.4840182648401826484018264840
TextTiling WD: 0.4840182648401826484018264840
SBERT PK: 0.04796163069544364508393285372
SBERT WD: 0.04796163069544364508393285372
